<a href="https://colab.research.google.com/github/LikithaSrinivasa25/ml-dl-algorithms/blob/main/machine-learning/random_forest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import re
import nltk
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [ ]:
df = pd.read_csv('comp1804_coursework_dataset_24-25.csv')
df.head()

,petition_ID,has_entity,relevant_department,deviation_across_regions,petition_status,petition_topic,petition_importance,petition_text
0,607158,EVENT:NO_DATE:NO_PERSON:NO_,Department for Education,0.000137,unsuccessful,education,not_important,Ban school uniforms in English schools. I want...
1,590629,EVENT:NO_DATE:YES_PERSON:NO_,Department of Health and Social Care,0.002133,unsuccessful,uk government and devolution,important,Lock NHS pay rises to increases in pay for MPs...
2,652724,EVENT:NO_DATE:NO_PERSON:NO_,Department of Health and Social Care,0.000272,unsuccessful,health and social care,important,Require dose counters on SABA metered dose inh...
3,629281,EVENT:NO_DATE:NO_PERSON:NO_,HM Treasury,0.000083,unsuccessful,"economy, labour and welfare",not_important,Remove the 20% VAT on private dementia day car...
4,301747,EVENT:NO_DATE:NO_PERSON:NO_,NaN,NaN,rejected,"culture, sport and media",not_important,Make bank holidays a holiday. Make bank holida...


In [ ]:
# Download NLTK resources (only need to run once)
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [ ]:

# Display basic information about the dataset
print(f"Original dataset shape: {df.shape}")

Original dataset shape: (8898, 8)


In [ ]:
# Check for null values
null_counts = df.isnull().sum()
print("\nNull values in each column:")
print(null_counts)


Null values in each column:
petition_ID                    0
has_entity                     0
relevant_department         1799
deviation_across_regions    1577
petition_status                0
petition_topic                31
petition_importance         8878
petition_text                  2
dtype: int64


In [ ]:
# Remove rows with null values in critical columns
df = df.dropna(subset=['petition_text', 'petition_topic'])

In [ ]:
# Handle missing values in 'has_entity' if present (replace with False)
if 'has_entity' in df.columns:
    df['has_entity'] = df['has_entity'].fillna(False)

In [ ]:
# Check for and remove duplicates
duplicate_count = df.duplicated(subset=['petition_text']).sum()
print(f"\nFound {duplicate_count} duplicate petitions")
df = df.drop_duplicates(subset=['petition_text'])


Found 404 duplicate petitions


In [ ]:
# Display cleaned dataset info
print(f"\nCleaned dataset shape: {df.shape}")
print("Missing values after cleaning:")
print(df.isnull().sum())


Cleaned dataset shape: (8461, 8)
Missing values after cleaning:
petition_ID                    0
has_entity                     0
relevant_department         1784
deviation_across_regions    1570
petition_status                0
petition_topic                 0
petition_importance         8441
petition_text                  0
dtype: int64


In [ ]:
# Text preprocessing function
def preprocess_text(text):
    if pd.isna(text) or not isinstance(text, str):
        return ""

    # Convert to lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)

    # Remove HTML tags
    text = re.sub(r'<.*?>', '', text)

    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))

    # Remove numbers
    text = re.sub(r'\d+', '', text)

    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    # Tokenize
    tokens = nltk.word_tokenize(text)

    # Remove stopwords
    stop_words = set(stopwords.words('english'))
    tokens = [word for word in tokens if word not in stop_words]

    # Lemmatize
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(word) for word in tokens]

    # Rejoin tokens
    return ' '.join(tokens)

In [ ]:
import nltk

nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
# Apply text preprocessing
print("\nApplying text preprocessing...")
df['processed_text'] = df['petition_text'].apply(preprocess_text)


Applying text preprocessing...


In [ ]:
# Check for empty texts after preprocessing
empty_texts = df['processed_text'].str.strip() == ''
if empty_texts.any():
    print(f"\nWarning: {empty_texts.sum()} petitions have empty text after preprocessing")
    # Remove rows with empty text after preprocessing
    df = df[~empty_texts]

In [ ]:
# Define the features and target
X = df[['processed_text', 'has_entity']]
y = df['petition_topic']

In [ ]:
# Check class distribution
topic_distribution = df['petition_topic'].value_counts()
print("\nClass distribution:")
print(topic_distribution)
print("\nClass distribution (%):")
print((topic_distribution / len(df) * 100).round(2))


Class distribution:
petition_topic
environment and animal welfare    2275
health and social care            2026
economy, labour and welfare       1182
education                         1146
culture, sport and media          1010
uk government and devolution       716
london                              59
Health And Social Care              15
Environment And Animal Welfare      13
Economy, Labour And Welfare          6
Education                            5
Culture, Sport And Media             5
Uk Government And Devolution         2
London                               1
Name: count, dtype: int64

Class distribution (%):
petition_topic
environment and animal welfare    26.89
health and social care            23.95
economy, labour and welfare       13.97
education                         13.54
culture, sport and media          11.94
uk government and devolution       8.46
london                             0.70
Health And Social Care             0.18
Environment And Animal Welfare  

In [ ]:
# Split the data into training and testing sets (80/20 split)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
print(f"\nTraining set size: {X_train.shape[0]}")
print(f"Testing set size: {X_test.shape[0]}")


Training set size: 6768
Testing set size: 1693


In [ ]:
# Create a preprocessing and model pipeline
class FeatureTransformer:
    def __init__(self):
        self.tfidf = TfidfVectorizer(
            max_features=5000,
            ngram_range=(1, 2),
            min_df=2,  # Ignore terms that appear in less than 2 documents
            max_df=0.9  # Ignore terms that appear in more than 90% of documents
        )

    def fit(self, X, y=None):
        # Fit the TF-IDF vectorizer on the text data
        self.tfidf.fit(X['processed_text'])
        return self

    def transform(self, X):
        # Transform text using TF-IDF
        text_features = self.tfidf.transform(X['processed_text'])

        # Add entity information as a feature
        # Convert 'has_entity' to numerical representation (0 or 1)
        entity_features = X['has_entity'].apply(lambda x: 1 if x != 'EVENT:NO_DATE:NO_PERSON:NO_' and x != 'nan' else 0).values.reshape(-1, 1)

        # Return combined features
        return np.hstack((text_features.toarray(), entity_features))



In [ ]:
# Create and train the pipeline
pipeline = Pipeline([
    ('features', FeatureTransformer()),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced'))
])

In [ ]:
print("\nTraining model...")
pipeline.fit(X_train, y_train)


Training model...


Pipeline(steps=[('features',
                 <__main__.FeatureTransformer object at 0x781a122f19d0>),
                ('classifier',
                 RandomForestClassifier(class_weight='balanced',
                                        random_state=42))])

In [ ]:
# Make predictions
y_train_pred = pipeline.predict(X_train)
y_test_pred = pipeline.predict(X_test)


In [ ]:
# Calculate performance metrics
train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)

In [ ]:
# Print results
print(f"\nTraining accuracy: {train_accuracy:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")


Training accuracy: 1.0000
Test accuracy: 0.8116
